# training

In [3]:
import os
import gc
import logging
from pathlib import Path
from typing import Dict, Any, Tuple, Union, Optional

import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    average_precision_score, roc_curve, precision_recall_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

import time
import json

def select_threshold(y_true: pd.Series, y_prob: np.ndarray) -> Tuple[float, float]:
    """Calculates the optimal probability threshold for maximizing the F1 score."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    # Add epsilon to prevent division by zero
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
    best_idx = np.argmax(f1_scores)
    # precision_recall_curve returns thresholds array that is 1 element shorter
    return thresholds[best_idx], f1_scores[best_idx]


def setup_logger(log_file: Path) -> logging.Logger:
    """Configures dual-output logging (console and file)."""
    logger = logging.getLogger("XGBoostPipeline")
    logger.setLevel(logging.INFO)
    
    if logger.hasHandlers():
        logger.handlers.clear()
        
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    
    file_handler = logging.FileHandler(log_file)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    
    return logger


class SugarcaneXGBoostPipeline:
    def __init__(
        self, 
        data_path: Union[str, Path], 
        output_dir: Union[str, Path],
        model_filename: str = "xgboost_sugarcane_model.json",
        test_size: float = 0.1,
        cv_folds: int = 3,
        optuna_trials: int = 30,
        optimize_metric: str = "auc",
        search_space: Optional[Dict[str, Any]] = None,
        final_train_sample_size: Optional[int] = None
    ):
        self.data_path = Path(data_path)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        self.model_filename = model_filename
        self.test_size = test_size
        self.cv_folds = cv_folds
        self.optuna_trials = optuna_trials
        self.optimize_metric = optimize_metric.lower()
        self.final_train_sample_size = final_train_sample_size
        
        self.log_file = self.output_dir / "training_pipeline.log"
        self.logger = setup_logger(self.log_file)
        
        self.fold_metrics_file = self.output_dir / "optuna_fold_metrics_tracking.csv"
        
        self.features = []
        self.target = 'label'
        self.label_map = {1: 1, 4: 0}
        
        self.search_space = search_space or {
            "learning_rate": {"type": "float", "low": 0.01, "high": 0.3, "log": True},
            "max_depth": {"type": "int", "low": 5, "high": 12},
            "min_child_weight": {"type": "int", "low": 1, "high": 10},
            "subsample": {"type": "float", "low": 0.6, "high": 1.0, "log": False},
            "colsample_bytree": {"type": "float", "low": 0.6, "high": 1.0, "log": False},
            "gamma": {"type": "float", "low": 0.0, "high": 5.0, "log": False},
            "lambda": {"type": "float", "low": 1e-3, "high": 10.0, "log": True},
            "alpha": {"type": "float", "low": 1e-3, "high": 10.0, "log": True},
        }
        
        valid_metrics = ["auc", "f1", "precision", "recall", "accuracy"]
        if self.optimize_metric not in valid_metrics:
            raise ValueError(f"optimize_metric must be one of {valid_metrics}")

    def load_and_prepare_data(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
        """Loads data, remaps labels, downcasts for memory, and splits train/test."""
        self.logger.info(f"Loading dataset from: {self.data_path.name}")
        df = pd.read_parquet(self.data_path, engine="pyarrow")
        
        self.logger.info("Remapping target labels (4 -> 0, 1 -> 1) for XGBoost strict binary compliance.")
        df[self.target] = df[self.target].map(self.label_map)
        
        self.features = [col for col in df.columns if col != self.target]
        
        self.logger.info("Downcasting numeric features to float32 to optimize memory footprint...")
        for col in self.features:
            if df[col].dtype == 'float64':
                df[col] = df[col].astype('float32')
                
        X = df[self.features]
        y = df[self.target]
        
        self.logger.info(f"Splitting data into {100-self.test_size*100}% Train and {self.test_size*100}% Final Evaluation (Test)...")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self.test_size, stratify=y, random_state=42
        )
        
        del df, X, y
        gc.collect()
        
        return X_train, X_test, y_train, y_test

    def _suggest_parameters(self, trial: optuna.Trial) -> Dict[str, Any]:
        """Dynamically parses the user-defined search space dictionary for Optuna."""
        params = {}
        for param_name, config in self.search_space.items():
            if config["type"] == "int":
                params[param_name] = trial.suggest_int(param_name, config["low"], config["high"])
            elif config["type"] == "float":
                params[param_name] = trial.suggest_float(
                    param_name, config["low"], config["high"], log=config.get("log", False)
                )
        return params

    def optimize_hyperparameters(self, X_train: pd.DataFrame, y_train: pd.Series, sample_size: int = 2000000) -> Dict[str, Any]:
        """Runs Bayesian Optimization tracking per-class metrics across folds in real-time."""
        self.logger.info(f"Initiating Bayesian Optimization ({self.optuna_trials} trials, {self.cv_folds} folds/trial)...")
        self.logger.info(f"Target metric for optimization: {self.optimize_metric.upper()}")
        
        if self.fold_metrics_file.exists():
            self.fold_metrics_file.unlink()
        
        if len(X_train) > sample_size:
            self.logger.info(f"Subsampling {sample_size:,} rows for optimization to ensure reasonable memory states.")
            X_opt, _, y_opt, _ = train_test_split(
                X_train, y_train, train_size=sample_size, stratify=y_train, random_state=42
            )
        else:
            X_opt, y_opt = X_train, y_train

        scale_pos_weight = y_opt.value_counts()[0] / y_opt.value_counts()[1]
        
        def objective(trial: optuna.Trial) -> float:
            param = self._suggest_parameters(trial)
            
            param.update({
                "objective": "binary:logistic",
                "eval_metric": "auc",
                "tree_method": "hist",
                "random_state": 42,
                "n_jobs": -1,
                "scale_pos_weight": scale_pos_weight,
            })
            
            cv = StratifiedKFold(n_splits=self.cv_folds, shuffle=True, random_state=42)
            target_metric_scores = []
            
            for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_opt, y_opt), 1):
                X_tr, y_tr = X_opt.iloc[train_idx], y_opt.iloc[train_idx]
                X_va, y_va = X_opt.iloc[val_idx], y_opt.iloc[val_idx]
                
                dtrain = xgb.DMatrix(X_tr, label=y_tr)
                dvalid = xgb.DMatrix(X_va, label=y_va)
                
                model = xgb.train(
                    param, 
                    dtrain, 
                    num_boost_round=200, 
                    evals=[(dvalid, 'eval')], 
                    early_stopping_rounds=20,
                    verbose_eval=False
                )
                
                preds_prob = model.predict(dvalid)
                preds_binary = (preds_prob >= 0.5).astype(int)
                
                auc = roc_auc_score(y_va, preds_prob)
                acc = accuracy_score(y_va, preds_binary)
                prec = precision_score(y_va, preds_binary, average=None, zero_division=0)
                rec = recall_score(y_va, preds_binary, average=None, zero_division=0)
                f1 = f1_score(y_va, preds_binary, average=None, zero_division=0)
                
                if self.optimize_metric == "auc":
                    target_score = auc
                elif self.optimize_metric == "accuracy":
                    target_score = acc
                elif self.optimize_metric == "f1":
                    target_score = f1[1]
                elif self.optimize_metric == "precision":
                    target_score = prec[1]
                elif self.optimize_metric == "recall":
                    target_score = rec[1]
                
                target_metric_scores.append(target_score)
                
                fold_results = {
                    "Trial_ID": trial.number,
                    "Fold": fold_idx,
                    "Target_Metric": target_score,
                    "AUC": auc,
                    "Accuracy": acc,
                    "Precision_Class0_NonCane": prec[0],
                    "Precision_Class1_Cane": prec[1],
                    "Recall_Class0_NonCane": rec[0],
                    "Recall_Class1_Cane": rec[1],
                    "F1_Class0_NonCane": f1[0],
                    "F1_Class1_Cane": f1[1]
                }
                fold_results.update({f"param_{k}": v for k, v in param.items() if k not in ["objective", "eval_metric", "tree_method", "random_state", "n_jobs"]})
                
                self.logger.info(f"Trial {trial.number} | Fold {fold_idx}/{self.cv_folds} | {self.optimize_metric.upper()}: {target_score:.4f} | F1_Cane: {f1[1]:.4f} | Prec_Cane: {prec[1]:.4f}")
                
                df_fold = pd.DataFrame([fold_results])
                write_header = not self.fold_metrics_file.exists()
                df_fold.to_csv(self.fold_metrics_file, mode='a', header=write_header, index=False)

            return float(np.mean(target_metric_scores))

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=self.optuna_trials)
        
        self.logger.info(f"Optimization Complete. Best Trial {self.optimize_metric.upper()}: {study.best_value:.4f}")
        self.logger.info(f"Best Parameters: {study.best_params}")
        
        best_params = study.best_params
        best_params.update({
            "objective": "binary:logistic",
            "tree_method": "hist",
            "scale_pos_weight": scale_pos_weight,
            "random_state": 42,
            "n_jobs": -1
        })
        
        del X_opt, y_opt
        gc.collect()
        
        return best_params

def train_final_model(self, X_train: pd.DataFrame, y_train: pd.Series, params: Dict[str, Any]) -> Tuple[xgb.Booster, float]:
        self.logger.info("Initializing Final Full-Scale Training...")

        # Internally carve out validation set so execute() doesn't break
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train, y_train, test_size=0.05, stratify=y_train, random_state=42
        )

        spw = float((y_tr == 0).sum() / (y_tr == 1).sum())
        params = {**params, "scale_pos_weight": spw}
        
        max_rounds = 600
        early_stopping = 30
        
        self.logger.info(f"Final training on {len(y_tr):,} rows | max_rounds={max_rounds} | early_stop={early_stopping}")

        t0 = time.time()
        dtrain = xgb.QuantileDMatrix(X_tr, label=y_tr)
        dval = xgb.QuantileDMatrix(X_val, label=y_val, ref=dtrain)
        self.logger.info(f"QuantileDMatrix built in {time.time() - t0:.1f}s")

        t0 = time.time()
        model = xgb.train(
            params, 
            dtrain, 
            num_boost_round=max_rounds, 
            evals=[(dval, "val")],
            early_stopping_rounds=early_stopping, 
            verbose_eval=50
        )
        self.logger.info(f"Trained in {time.time() - t0:.1f}s | best_iteration={model.best_iteration} | val {params['eval_metric']}={model.best_score:.4f}")

        # Get optimal threshold dynamically
        p_val = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
        threshold, f1_val = select_threshold(y_val, p_val)

        model_path = self.output_dir / self.model_filename
        model.save_model(str(model_path))
        
        config = {
            "model_file": self.model_filename,
            "features": self.features,
            "label_map": {str(k): v for k, v in self.label_map.items()},
            "threshold": float(threshold),
            "best_iteration": int(model.best_iteration),
            "val_f1_cane_at_threshold": float(f1_val),
            "val_pr_auc": float(average_precision_score(y_val, p_val)),
            "params": {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in params.items()},
        }
        
        with open(self.output_dir / "inference_config.json", "w") as f:
            json.dump(config, f, indent=2)
            
        self.logger.info(f"Saved {model_path.name} + inference_config.json | threshold={threshold:.4f} (val F1_cane={f1_val:.4f})")

        del dtrain, dval, X_tr, X_val, y_tr, y_val
        gc.collect()
        
        return model, float(threshold)

    def evaluate_and_plot(self, model: xgb.Booster, X_test: pd.DataFrame, y_test: pd.Series, optimal_threshold: float) -> None:
        self.logger.info("Evaluating final model on isolated holdout set...")
        
        dtest = xgb.DMatrix(X_test, label=y_test)
        y_prob = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))
        
        # Apply the optimal threshold found during early stopping instead of default 0.5
        y_pred = (y_prob >= optimal_threshold).astype(int)
        
        self.logger.info(f"--- Final Holdout Evaluation Metrics (Threshold: {optimal_threshold:.4f}) ---")
        report = classification_report(y_test, y_pred, target_names=["Non-Cane (0)", "Cane (1)"])
        self.logger.info("\n" + report)
        
        with open(self.output_dir / "final_classification_report.txt", "w") as f:
            f.write(report)

        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Non-Cane", "Cane"], yticklabels=["Non-Cane", "Cane"])
        plt.title("Confusion Matrix (Final Model)")
        plt.ylabel("Actual")
        plt.xlabel("Predicted")
        plt.tight_layout()
        plt.savefig(self.output_dir / "confusion_matrix.png", dpi=300)
        plt.close()

        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_score(y_test, y_prob):.4f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic (ROC)')
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(self.output_dir / "roc_curve.png", dpi=300)
        plt.close()

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        xgb.plot_importance(model, importance_type='weight', ax=axes[0], title='Feature Importance (Weight)', show_values=False)
        xgb.plot_importance(model, importance_type='gain', ax=axes[1], title='Feature Importance (Gain)', show_values=False)
        plt.tight_layout()
        plt.savefig(self.output_dir / "feature_importance_xgb.png", dpi=300)
        plt.close()
        
        self.logger.info(f"All evaluation plots and reports saved to {self.output_dir}")

    def execute(self) -> None:
        X_train, X_test, y_train, y_test = self.load_and_prepare_data()
        best_params = self.optimize_hyperparameters(X_train, y_train)
        
        # Unpack both the model and the optimal threshold from the training phase
        final_model, optimal_threshold = self.train_final_model(X_train, y_train, best_params)
        
        self.evaluate_and_plot(final_model, X_test, y_test, optimal_threshold)
        self.logger.info("Pipeline Execution Terminated Successfully.")


In [4]:

if __name__ == "__main__":
    MASTER_DATA_PATH = "/home/jovyan/FAO/cane/static_model_v3/try_1/master_training_data_25Aug2026.parquet"
    OUTPUT_DIRECTORY = r"/home/jovyan/FAO/cane/static_model_v3/try_1/model_training_outputs_v1"
    
    custom_space = {
        "learning_rate": {"type": "float", "low": 0.01, "high": 0.2, "log": True},
        "max_depth": {"type": "int", "low": 4, "high": 10},
        "min_child_weight": {"type": "int", "low": 1, "high": 8},
        "subsample": {"type": "float", "low": 0.7, "high": 1.0, "log": False},
        "colsample_bytree": {"type": "float", "low": 0.7, "high": 1.0, "log": False},
        "gamma": {"type": "float", "low": 0.0, "high": 3.0, "log": False}
    }
    
    pipeline = SugarcaneXGBoostPipeline(
        data_path=MASTER_DATA_PATH,
        output_dir=OUTPUT_DIRECTORY,
        model_filename="xgb_cane_model_v3_opt.json", 
        test_size=0.1,                                  # Fully parametrized holdout split
        cv_folds=2,                                    
        optuna_trials=25,                             
        optimize_metric="f1",                         
        search_space=custom_space,                     
        final_train_sample_size=10_000_000              # Cap execution overhead. Set to None to use all data.
    )
    
    pipeline.execute()

2026-08-25 06:52:37,039 - INFO - Loading dataset from: master_training_data_25Aug2026.parquet
2026-08-25 06:52:40,825 - INFO - Remapping target labels (4 -> 0, 1 -> 1) for XGBoost strict binary compliance.
2026-08-25 06:52:42,265 - INFO - Downcasting numeric features to float32 to optimize memory footprint...
2026-08-25 06:52:43,772 - INFO - Splitting data into 90.0% Train and 10.0% Final Evaluation (Test)...
2026-08-25 06:53:31,981 - INFO - Initiating Bayesian Optimization (25 trials, 2 folds/trial)...
2026-08-25 06:53:31,982 - INFO - Target metric for optimization: F1
2026-08-25 06:53:31,983 - INFO - Subsampling 2,000,000 rows for optimization to ensure reasonable memory states.
[I 2026-08-25 06:54:27,436] A new study created in memory with name: no-name-0096a8f7-42c6-4e90-86e4-a9c5f45de797
2026-08-25 06:56:52,396 - INFO - Trial 0 | Fold 1/2 | F1: 0.8550 | F1_Cane: 0.8550 | Prec_Cane: 0.7779
2026-08-25 06:59:23,641 - INFO - Trial 0 | Fold 2/2 | F1: 0.8550 | F1_Cane: 0.8550 | Prec_Can

TypeError: SugarcaneXGBoostPipeline.train_final_model() missing 2 required positional arguments: 'y_val' and 'params'

# backup

In [1]:
import os
import gc
import logging
import json
import time
from pathlib import Path
from typing import Dict, Any, Tuple, Union

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    average_precision_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns


def setup_logger(log_file: Path) -> logging.Logger:
    logger = logging.getLogger("XGBoostRecovery")
    logger.setLevel(logging.INFO)
    if logger.hasHandlers():
        logger.handlers.clear()
        
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    
    file_handler = logging.FileHandler(log_file)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    
    return logger


class XGBoostRecoveryPipeline:
    def __init__(
        self, 
        data_path: Union[str, Path], 
        output_dir: Union[str, Path],
        best_params: Dict[str, Any],
        model_filename: str = "xgb_cane_model_v3_opt.json",
        test_size: float = 0.1,
        val_size: float = 0.05,
        final_train_sample_size: int = 10_000_000
    ):
        self.data_path = Path(data_path)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        self.best_params = best_params
        self.model_filename = model_filename
        self.test_size = test_size
        self.val_size = val_size
        self.final_train_sample_size = final_train_sample_size
        
        self.log_file = self.output_dir / "recovery_training.log"
        self.logger = setup_logger(self.log_file)
        
        self.features = []
        self.target = 'label'
        self.label_map = {1: 1, 4: 0}

    def load_and_prepare_data(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
        self.logger.info(f"Loading dataset from: {self.data_path.name}")
        df = pd.read_parquet(self.data_path, engine="pyarrow")
        
        self.logger.info("Remapping target labels (4 -> 0, 1 -> 1).")
        df[self.target] = df[self.target].map(self.label_map)
        
        self.features = [col for col in df.columns if col != self.target]
        
        self.logger.info("Downcasting numeric features to float32...")
        for col in self.features:
            if df[col].dtype == 'float64':
                df[col] = df[col].astype('float32')
                
        X = df[self.features]
        y = df[self.target]
        
        self.logger.info(f"Isolating {self.test_size*100}% Holdout Test Set...")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self.test_size, stratify=y, random_state=42
        )
        
        del df, X, y
        gc.collect()
        
        return X_train, X_test, y_train, y_test

    def train_final_model(self, X_train: pd.DataFrame, y_train: pd.Series) -> xgb.Booster:
        self.logger.info("Initializing Final Full-Scale Training...")
        
        if self.final_train_sample_size and len(X_train) > self.final_train_sample_size:
            self.logger.info(f"Subsampling training data to {self.final_train_sample_size:,} rows.")
            X_train, _, y_train, _ = train_test_split(
                X_train, y_train, train_size=self.final_train_sample_size, stratify=y_train, random_state=42
            )
            
        self.logger.info(f"Extracting {self.val_size*100}% internal validation set for Early Stopping...")
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train, y_train, test_size=self.val_size, stratify=y_train, random_state=42
        )
        
        scale_pos_weight = float((y_tr == 0).sum() / (y_tr == 1).sum())
        self.best_params["scale_pos_weight"] = scale_pos_weight
        
        self.logger.info("Constructing QuantileDMatrix objects...")
        d_tr = xgb.QuantileDMatrix(X_tr, label=y_tr)
        d_va = xgb.QuantileDMatrix(X_val, label=y_val, ref=d_tr)
        
        self.logger.info("Training Booster... (Monitoring via Early Stopping)")
        t0 = time.time()
        final_model = xgb.train(
            self.best_params,
            d_tr,
            num_boost_round=600,
            evals=[(d_tr, 'train'), (d_va, 'eval')],
            early_stopping_rounds=30,
            verbose_eval=50
        )
        self.logger.info(f"Training completed in {time.time() - t0:.1f}s | best_iteration={final_model.best_iteration}")
        
        model_path = self.output_dir / self.model_filename
        final_model.save_model(str(model_path))
        self.logger.info(f"Final model serialized to {model_path}")
        
        del d_tr, d_va, X_tr, X_val, y_tr, y_val
        gc.collect()
        
        return final_model

    def evaluate_and_plot(self, model: xgb.Booster, X_test: pd.DataFrame, y_test: pd.Series) -> None:
        self.logger.info("Evaluating final model on isolated holdout set...")
        
        dtest = xgb.DMatrix(X_test, label=y_test)
        
        # Use iteration_range to ensure we predict using the optimal tree cutoff from early stopping
        y_prob = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))
        y_pred = (y_prob >= 0.5).astype(int)
        
        report = classification_report(y_test, y_pred, target_names=["Non-Cane (0)", "Cane (1)"])
        self.logger.info("\n" + report)
        
        with open(self.output_dir / "final_classification_report.txt", "w") as f:
            f.write(report)

        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Non-Cane", "Cane"], yticklabels=["Non-Cane", "Cane"])
        plt.title("Confusion Matrix (Final Model)")
        plt.ylabel("Actual")
        plt.xlabel("Predicted")
        plt.tight_layout()
        plt.savefig(self.output_dir / "confusion_matrix.png", dpi=300)
        plt.close()

        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_score(y_test, y_prob):.4f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic (ROC)')
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(self.output_dir / "roc_curve.png", dpi=300)
        plt.close()

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        xgb.plot_importance(model, importance_type='weight', ax=axes[0], title='Feature Importance (Weight)', show_values=False)
        xgb.plot_importance(model, importance_type='gain', ax=axes[1], title='Feature Importance (Gain)', show_values=False)
        plt.tight_layout()
        plt.savefig(self.output_dir / "feature_importance_xgb.png", dpi=300)
        plt.close()
        
        self.logger.info(f"All evaluation plots saved to {self.output_dir}")

    def execute(self) -> None:
        X_train, X_test, y_train, y_test = self.load_and_prepare_data()
        final_model = self.train_final_model(X_train, y_train)
        self.evaluate_and_plot(final_model, X_test, y_test)
        self.logger.info("Recovery Pipeline Execution Terminated Successfully.")


if __name__ == "__main__":
    MASTER_DATA_PATH = "/home/jovyan/FAO/cane/static_model_v3/try_1/master_training_data_25Aug2026.parquet"
    OUTPUT_DIRECTORY = r"/home/jovyan/FAO/cane/static_model_v3/try_1/model_training_outputs_v1"
    
    # Injected parameters extracted directly from the successful Trial 21 log output
    WINNING_PARAMS = {
        'learning_rate': 0.06136567060443863, 
        'max_depth': 10, 
        'min_child_weight': 4, 
        'subsample': 0.9189786443226184, 
        'colsample_bytree': 0.9806194772432495, 
        'gamma': 1.6413232382360872,
        'objective': 'binary:logistic',
        'tree_method': 'hist',
        'eval_metric': 'auc',
        'random_state': 42,
        'n_jobs': -1
    }
    
    recovery_pipeline = XGBoostRecoveryPipeline(
        data_path=MASTER_DATA_PATH,
        output_dir=OUTPUT_DIRECTORY,
        best_params=WINNING_PARAMS,
        model_filename="xgb_cane_model_v3_opt.json",
        test_size=0.1,
        final_train_sample_size=10_000_000
    )
    
    recovery_pipeline.execute()

2026-08-25 09:15:26,468 - INFO - Loading dataset from: master_training_data_25Aug2026.parquet
2026-08-25 09:15:29,159 - INFO - Remapping target labels (4 -> 0, 1 -> 1).
2026-08-25 09:15:29,842 - INFO - Downcasting numeric features to float32...
2026-08-25 09:15:30,703 - INFO - Isolating 10.0% Holdout Test Set...
2026-08-25 09:15:55,610 - INFO - Initializing Final Full-Scale Training...
2026-08-25 09:15:55,611 - INFO - Subsampling training data to 10,000,000 rows.
2026-08-25 09:16:24,903 - INFO - Extracting 5.0% internal validation set for Early Stopping...
2026-08-25 09:16:29,256 - INFO - Constructing QuantileDMatrix objects...
2026-08-25 09:16:36,785 - INFO - Training Booster... (Monitoring via Early Stopping)


[0]	train-auc:0.97557	eval-auc:0.97536
[50]	train-auc:0.98143	eval-auc:0.98105
[100]	train-auc:0.98244	eval-auc:0.98187
[150]	train-auc:0.98271	eval-auc:0.98206
[200]	train-auc:0.98292	eval-auc:0.98218
[250]	train-auc:0.98306	eval-auc:0.98224
[300]	train-auc:0.98316	eval-auc:0.98227
[350]	train-auc:0.98327	eval-auc:0.98230
[400]	train-auc:0.98335	eval-auc:0.98232
[450]	train-auc:0.98343	eval-auc:0.98233
[500]	train-auc:0.98349	eval-auc:0.98234
[550]	train-auc:0.98355	eval-auc:0.98235
[599]	train-auc:0.98361	eval-auc:0.98236


2026-08-25 09:43:33,144 - INFO - Training completed in 1616.4s | best_iteration=597
2026-08-25 09:43:33,426 - INFO - Final model serialized to /home/jovyan/FAO/cane/static_model_v3/try_1/model_training_outputs_v1/xgb_cane_model_v3_opt.json
2026-08-25 09:43:33,548 - INFO - Evaluating final model on isolated holdout set...
2026-08-25 09:45:07,268 - INFO - 
              precision    recall  f1-score   support

Non-Cane (0)       0.99      0.92      0.95   4000189
    Cane (1)       0.78      0.95      0.86   1131984

    accuracy                           0.93   5132173
   macro avg       0.88      0.94      0.91   5132173
weighted avg       0.94      0.93      0.93   5132173

2026-08-25 09:45:13,159 - INFO - All evaluation plots saved to /home/jovyan/FAO/cane/static_model_v3/try_1/model_training_outputs_v1
2026-08-25 09:45:13,181 - INFO - Recovery Pipeline Execution Terminated Successfully.
